# ScreamingFace ↔ URL4 engine

Combine three model routes into one URL4-backed fusion, run it on deterministic benchmark
questions, and measure whether the majority beats the best individual model.

**Path: architecture walkthrough.** This notebook deliberately exposes the recipe, concrete
expression, approximate compiled node tree, and response contract. For the small public API path,
start with [`00_quickstart.ipynb`](00_quickstart.ipynb).

By default, this uses a real URL4 node in-process with deterministic model-route leaves, so the
whole notebook runs without services or credentials. To exercise the same expressions over HTTP,
start the optional development engine from `packages/screamingface`:

```bash
./scripts/dev-url4.sh
```

To fetch GPQA Diamond instead of the bundled fixture, first accept its gated dataset terms and be
logged in to Hugging Face, then select the live dataset with `sf.config(mode="live")`. Your URL4
engine only needs production-backed routes when you explicitly select an HTTP engine.

The saved run uses deterministic routes, so its result is reproducible and makes no
provider-quality claim.

## 1 · Import the SDK

In [ ]:
import screamingface as sf

# Optional: send the same expressions over HTTP instead of running URL4 in-process.
# sf.config("http://127.0.0.1:4404")  # first run ./scripts/dev-url4.sh
# sf.config("https://url4.example")

## 2 · Compose a fusion — Python or YAML

These are two representations of the same fusion. Use Python while exploring; use YAML when you
want a small configuration file to review or share.

### Option A · Python

In [ ]:
fusion = sf.Fusion(
    "frontier-trio",
    models=[
        "codex/gpt-5.5",
        "gemini-cli/gemini-2.5-pro",
        "anthropic/claude-sonnet-4-6",
    ],
    reducer=sf.MajorityVote(tie_breaker="codex/gpt-5.5"),
)
fusion

A plain string is shorthand for `{"model": "provider/model"}`. Mix in a strict
dictionary only when one model needs its own `name`, `prompt`, or URL4 `params`:

```python
fusion = sf.Fusion(
    "research-panel",
    prompt="Answer the benchmark question carefully: $question",
    models=[
        "openai/gpt-5.5",
        {
            "model": "anthropic/claude-opus-4.8",
            "name": "opus-sample-1",
            "prompt": "Independently solve and check this question: $question",
            "params": {"temperature": 0.7},
        },
    ],
)
```

`Fusion(prompt=...)` supplies the shared panel prompt. A model dictionary's `prompt` overrides it
for that call; if neither is supplied, the prompt is `$question`. ScreamingFace validates these
dictionaries and assigns private call-slot identities. There is no public `Member` or `Source`
wrapper. Reducers use typed objects because they select executable behavior, but panel calls and
model reducers share the same validated route, prompt, and parameter semantics internally.

### Option B · YAML

The equivalent [`fusion.yaml`](fusion.yaml) is:

```yaml
name: frontier-trio
models:
  - codex/gpt-5.5
  - gemini-cli/gemini-2.5-pro
  - anthropic/claude-sonnet-4-6
reducer:
  kind: majority_vote
  tie_breaker: codex/gpt-5.5
```

In [ ]:
fusion_from_yaml = sf.Fusion.from_yaml("fusion.yaml")
fusion_from_yaml.url4 == fusion.url4

## 3 · Inspect the shareable recipe

The recipe contains model routes and an unresolved `$question`. Constructing or displaying it
sends nothing. Evaluation binds each concrete question later.

In [ ]:
fusion.url4

## 4 · Run through the URL4 engine

The default in-process URL4 node parses the complete expression, executes all three deterministic
model routes, and returns their labeled answers without network I/O. When an HTTP engine is
selected, ScreamingFace sends that same expression to `/v1`; it never calls model routes,
AI Gateway, or providers directly.

In [ ]:
# HTTP-engine equivalent:
# GET http://127.0.0.1:4404/v1?q=<URL-encoded fusion expression>
# Decoded q expression:
# (question='<resolved GPQA prompt>',
#  panel_1=/codex/gpt-5.5()!'$question',
#  panel_2=/gemini/2.5()!'$question',
#  panel_3=/claude/sonnet-4.6()!'$question',
#  {schema: 'screamingface.panel-result.v2',
#   panel_1_id: 'codex/gpt-5.5', panel_1_model: 'codex/gpt-5.5',
#   panel_1_answer: '$panel_1',
#   panel_2_id: 'gemini-cli/gemini-2.5-pro',
#   panel_2_model: 'gemini-cli/gemini-2.5-pro', panel_2_answer: '$panel_2',
#   panel_3_id: 'anthropic/claude-sonnet-4-6',
#   panel_3_model: 'anthropic/claude-sonnet-4-6', panel_3_answer: '$panel_3'})
#
# Compiled URL4 request node (↖ shared = the same binding, not another request):
# GatherNode
# ├─ question: BindingNode → TextNode '<resolved GPQA prompt>'
# ├─ panel_1: BindingNode → RelUrlNode /codex/gpt-5.5
# │  ├─ context → empty
# │  └─ intent → question ↖ shared
# ├─ panel_2: BindingNode → RelUrlNode /gemini/2.5
# │  ├─ context → empty
# │  └─ intent → question ↖ shared
# ├─ panel_3: BindingNode → RelUrlNode /claude/sonnet-4.6
# │  ├─ context → empty
# │  └─ intent → question ↖ shared
# └─ response: StructNode
#    ├─ schema → screamingface.panel-result.v2
#    ├─ panel_1_id → codex/gpt-5.5
#    ├─ panel_1_model → codex/gpt-5.5
#    ├─ panel_1_answer → panel_1 ↖ shared
#    ├─ panel_2_id → gemini-cli/gemini-2.5-pro
#    ├─ panel_2_model → gemini-cli/gemini-2.5-pro
#    ├─ panel_2_answer → panel_2 ↖ shared
#    ├─ panel_3_id → anthropic/claude-sonnet-4-6
#    ├─ panel_3_model → anthropic/claude-sonnet-4-6
#    └─ panel_3_answer → panel_3 ↖ shared
run = fusion.evaluate("gpqa", first=20, seed=0)
run

### What URL4 returns

The engine returns the evaluated final struct as JSON text. The default deterministic node and an
HTTP production node must use the same labeled envelope:

```json
{
  "schema": "screamingface.panel-result.v2",
  "panel_1_id": "codex/gpt-5.5",
  "panel_1_model": "codex/gpt-5.5",
  "panel_1_answer": "A",
  "panel_2_id": "gemini-cli/gemini-2.5-pro",
  "panel_2_model": "gemini-cli/gemini-2.5-pro",
  "panel_2_answer": "B",
  "panel_3_id": "anthropic/claude-sonnet-4-6",
  "panel_3_model": "anthropic/claude-sonnet-4-6",
  "panel_3_answer": "A"
}
```

The letters vary by question. The slot IDs and models do not: ScreamingFace validates that
association before voting and scoring. The mock is only in the route handlers that supplied these
answers; URL4 still parsed the expression, resolved `$question`, executed the graph, and built this
response.

## 5 · Compare

In [ ]:
{
    "sample_size": run.sample_size,
    "score": run.score,
    "baseline": run.baseline,
    "gain": run.gain,
}

> `gain = fusion score − best model score` on the same answers. Positive gain
means the combination corrected mistakes made by every individual panel model.

The URL4 engine owns model execution. ScreamingFace owns majority vote, answer-key scoring,
baseline, and gain. Real AI-Gateway-backed model routes can replace the deterministic commands
later without changing this SDK flow.

## 6 · Replace voting with one model-backed reducer

`Reducer` is the common contract; users construct concrete mechanisms. `MajorityVote` runs
deterministically in the SDK. `ModelReducer` adds one later URL4 model call whose prompt can
synthesize, select, rank, merge, or adjudicate the resolved panel answers.

The `$panel_answers` binding contains stable private slot IDs, model IDs, and resolved answers. The
reducer receives it in its URL4 intent, with empty context.

In [ ]:
model_reduced = sf.Fusion(
    "frontier-trio-model-reduced",
    models=fusion.models,
    reducer=sf.ModelReducer(
        model="codex/gpt-5.5",
        prompt="Synthesize one final answer for $question from $panel_answers",
        params={"temperature": 0.0, "max_tokens": 512},
    ),
)
model_reduced.url4

In [ ]:
model_run = model_reduced.evaluate("gpqa", first=3, seed=0)
model_run

A model-backed reducer changes the envelope to
`screamingface.fusion-result.v2`. It retains every labeled panel answer and adds `reducer`,
`reducer_model`, and the final `answer`. See the complete field-level reference in
[`../docs/index.html`](../docs/index.html#wire).